# Notebook 5: Feature Engineering and School District Join (Week 6)

This notebook loads `housing_after_type_parsing.csv`, the full pre-split checkpoint from notebook 2 (added as a one-line save immediately after type parsing, before the date-scope filter). Working from the pre-split dataset matters here because the new implausible-value cutoff and the school district join are both one-time, deterministic decisions about the data itself, not statistics fit to any particular partition. They should happen once, before a fresh chronological split, not be inherited from the m1 split's already-filtered output.

This notebook produces a second, independent set of partitions named with an `m2` suffix (`housingtrainm2`, `housingvalm2`, `housingtestm2`), matching the m1 naming convention used in notebooks 3 and 4.

One thing worth stating plainly before anything else: the school district join is only as good as the coordinates it runs on, and the Hemet placeholder cluster documented earlier is exactly the kind of problem that would silently mislabel every property in it into whichever district that fake point happens to fall inside. That issue was deferred to a later notebook. This is that notebook. A diagnostic cell below reports the cluster's size and its effect directly rather than letting it pass unnoticed. If the geocode confirmation pipeline built earlier has finished running, merging those confirmed coordinates in before this join would meaningfully improve it, but this notebook proceeds on the coordinates currently in the pipeline so it isn't blocked on that.

In [3]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

RANDOM_STATE = 42

## 1. Load Pre-Split Data

In [4]:
os.chdir(os.path.expanduser("~/Desktop/CAPropPredictor"))

housing = pd.read_csv("CRMLSCleaned/housing_after_type_parsing.csv")
housing["SaleYearMonth"] = pd.PeriodIndex(housing["SaleYearMonth"], freq="M")

print(f"loaded: {housing.shape}")

loaded: (320072, 25)


## 2. Systematic Implausible Bathroom Count Cutoff

The earlier idea of dropping anything above 20 bathrooms was an arbitrary round number picked without looking at the data. The same elbow reasoning used for the null rate threshold applies here, but the mechanism has to be different. Null rate is a bounded proportion, so a gap in the raw value works fine as a signal. Bathroom count is an open ended integer, and a naive largest gap in the raw value tends to find the wrong elbow, since gaps between large sparse numbers grow just from scale even when none of those numbers are meaningfully different from each other. Tested this directly against synthetic data with six implausible values injected: the raw value gap method caught only two of the six.

What actually distinguishes a plausible bathroom count from an implausible one is not the size of the number, it is how common it is. A four bathroom house is unremarkable and shows up hundreds of times. A ninety nine bathroom house is not a real single family home and shows up once. So the cutoff here is based on where per value frequency collapses, in log space so the comparison is proportional rather than absolute. That method caught all six injected anomalies in testing.

In [5]:
def find_frequency_cliff_cutoff(series, min_value_to_consider=1, max_removal_fraction=0.02):

    counts = series.value_counts().sort_index()
    counts = counts[counts.index >= min_value_to_consider]
    values = counts.index.to_numpy()
    log_counts = np.log(counts.to_numpy())
    drops = -np.diff(log_counts)
    steepest_drop_idx = np.argmax(drops)
    cutoff = values[steepest_drop_idx]

    total = counts.sum()
    would_remove = counts[counts.index > cutoff].sum()
    removal_fraction = would_remove / total
    is_safe = removal_fraction <= max_removal_fraction

    return cutoff, counts, is_safe, removal_fraction

bath_cutoff, bath_counts, bath_is_safe, bath_removal_fraction = find_frequency_cliff_cutoff(housing["BathroomsTotalInteger"])
print("bathroom value counts, tail end:")
print(bath_counts.tail(15))
print(f"\ncutoff (last value in the dense cluster): {bath_cutoff}")
print(f"would remove {bath_removal_fraction:.2%} of rows, flagged safe: {bath_is_safe}")

rows_before_bath_filter = len(housing)
if bath_is_safe:
    housing = housing[housing["BathroomsTotalInteger"] <= bath_cutoff]
    print(f"dropped {rows_before_bath_filter - len(housing)} rows for implausible bathroom count "
          f"({rows_before_bath_filter} -> {len(housing)})")
else:
    print("NOT applying this cutoff automatically -- removal fraction exceeds the safety "
          "threshold, which means this may not be a real anomaly boundary. Inspect the "
          "value counts above manually and set a cutoff by hand if one is warranted.")

bathroom value counts, tail end:
BathroomsTotalInteger
14.0     12
15.0      6
16.0      6
17.0      2
18.0      4
20.0      3
21.0      2
22.0      2
23.0      1
25.0      1
27.0      2
35.0      1
45.0      1
153.0     1
175.0     1
Name: count, dtype: int64

cutoff (last value in the dense cluster): 3.0
would remove 13.99% of rows, flagged safe: False
NOT applying this cutoff automatically -- removal fraction exceeds the safety threshold, which means this may not be a real anomaly boundary. Inspect the value counts above manually and set a cutoff by hand if one is warranted.


Same method applied to bedroom count. A separate call rather than reusing the bathroom cutoff value, since there is no reason the two should collapse at the same number.

In [6]:
bed_cutoff, bed_counts, bed_is_safe, bed_removal_fraction = find_frequency_cliff_cutoff(housing["BedroomsTotal"])
print("bedroom value counts, tail end:")
print(bed_counts.tail(15))
print(f"\ncutoff (last value in the dense cluster): {bed_cutoff}")
print(f"would remove {bed_removal_fraction:.2%} of rows, flagged safe: {bed_is_safe}")

rows_before_bed_filter = len(housing)
if bed_is_safe:
    housing = housing[housing["BedroomsTotal"] <= bed_cutoff]
    print(f"dropped {rows_before_bed_filter - len(housing)} rows for implausible bedroom count "
          f"({rows_before_bed_filter} -> {len(housing)})")
else:
    print("NOT applying this cutoff automatically -- removal fraction exceeds the safety "
          "threshold, which means this may not be a real anomaly boundary. Inspect the "
          "value counts above manually and set a cutoff by hand if one is warranted.")

bedroom value counts, tail end:
BedroomsTotal
8     296
9      84
10     43
11     14
12     13
13      6
14      3
15      4
16      2
17      1
19      1
22      1
31      1
34      1
45      1
Name: count, dtype: int64

cutoff (last value in the dense cluster): 5
would remove 2.21% of rows, flagged safe: False
NOT applying this cutoff automatically -- removal fraction exceeds the safety threshold, which means this may not be a real anomaly boundary. Inspect the value counts above manually and set a cutoff by hand if one is warranted.


Worth a manual look at whatever gets flagged before trusting it fully. A frequency cliff is strong evidence of a data entry error, but it is still just evidence, not certainty. A genuine 20 bathroom mega mansion is rare but not impossible, and this method has no way to distinguish that from a mistyped value on its own.

## 3. Engineered Features: Property Age and Bed/Bath Ratio

Property age is computed relative to the sale month rather than a fixed reference date, since the point is how old the house was at the time it sold, not how old it is today. Bed to bath ratio uses safe division, zero bathroom rows become null rather than infinite, so the imputer downstream handles them the same way it handles any other missing numeric value instead of choking on inf.

In [7]:
housing["PropertyAgeYears"] = housing["SaleYearMonth"].apply(lambda p: p.year) - housing["YearBuilt"]

housing["BedBathRatio"] = np.where(
    housing["BathroomsTotalInteger"] > 0,
    housing["BedroomsTotal"] / housing["BathroomsTotalInteger"],
    np.nan,
)

print(f"PropertyAgeYears range: [{housing['PropertyAgeYears'].min()}, {housing['PropertyAgeYears'].max()}]")
print(f"PropertyAgeYears nulls: {housing['PropertyAgeYears'].isna().sum()}")
print(f"BedBathRatio nulls (0-bathroom rows): {housing['BedBathRatio'].isna().sum()}")

PropertyAgeYears range: [-1.0, 249.0]
PropertyAgeYears nulls: 202
BedBathRatio nulls (0-bathroom rows): 149


A negative `PropertyAgeYears` would mean a sale recorded before the house was built, which is a logical impossibility in the same category as the checks from the earlier data quality section.

In [8]:
negative_age_mask = housing["PropertyAgeYears"] < 0
print(f"rows with negative PropertyAgeYears: {negative_age_mask.sum()}")
if negative_age_mask.sum() > 0:
    display(housing.loc[negative_age_mask, ["SaleYearMonth", "YearBuilt", "PropertyAgeYears"]].head(10))
    rows_before_age_filter = len(housing)
    housing = housing[~negative_age_mask]
    print(f"dropped {rows_before_age_filter - len(housing)} rows with negative property age")

rows with negative PropertyAgeYears: 27


,SaleYearMonth,YearBuilt,PropertyAgeYears
94084,2024-08,2025.0,-1.0
94550,2024-09,2025.0,-1.0
116823,2024-10,2025.0,-1.0
133011,2024-12,2025.0,-1.0
134255,2024-12,2025.0,-1.0
136841,2024-12,2025.0,-1.0
137294,2024-12,2025.0,-1.0
138203,2024-12,2025.0,-1.0
138514,2024-12,2025.0,-1.0
138728,2024-12,2025.0,-1.0


dropped 27 rows with negative property age


## 4. School District Spatial Join

The shapefile field names are inspected before anything is assumed about them. Shapefiles truncate field names to 10 characters, a limitation of the format itself, not something specific to this file, so a name like `DistrictName` in the source data almost certainly comes through as something shorter. Guessing the column name ahead of time risks silently joining on the wrong field or crashing on a KeyError. Printing the actual columns first avoids both.

In [9]:
districts_gdf = gpd.read_file("DistrictAreas2425/DistrictAreas2425.shp")
print("columns:", list(districts_gdf.columns))
print("CRS:", districts_gdf.crs)
print("row count:", len(districts_gdf))
districts_gdf.head()

columns: ['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName', 'DistrictNa', 'DistrictTy', 'GradeLow', 'GradeHigh', 'GradeLowCe', 'GradeHighC', 'AssistStat', 'CongressUS', 'SenateCA', 'AssemblyCA', 'UpdateNote', 'EnrollTota', 'EnrollChar', 'EnrollNonC', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAre', 'Shape__Are', 'Shape__Len', 'geometry']
CRS: EPSG:3857
row count: 937


,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictNa,DistrictTy,GradeLow,GradeHigh,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAre,Shape__Are,Shape__Len,geometry
0,1,2024-25,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0,0.0,1356,12.6,3958,36.7,11.248939,4.755489e+07,56522.982683,"MULTIPOLYGON (((-13606222.82 4540862.699, -136..."
1,2,2024-25,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0,0.0,357,9.7,1184,32.1,1.789984,7.096327e+06,12696.382797,"POLYGON ((-13612893.866 4565099.707, -13612896..."
2,3,2024-25,0604740,0161143,01611430000000,Alameda,Berkeley Unified,Unified,PK,12,...,0,0.0,1111,12.1,2686,29.3,10.434329,4.364648e+07,43695.341538,"POLYGON ((-13609482.48 4565074.597, -13609483...."
3,4,2024-25,0607800,0161150,01611500000000,Alameda,Castro Valley Unified,Unified,PK,12,...,0,0.0,1113,11.6,3728,39.0,66.885571,2.838285e+08,142492.767565,"MULTIPOLYGON (((-13582508.535 4529067.071, -13..."
4,5,2024-25,0612630,0161168,01611680000000,Alameda,Emery Unified,Unified,PK,12,...,0,0.0,81,13.7,412,69.8,1.273929,5.363392e+06,13741.272894,"POLYGON ((-13613999.038 4555592.769, -13614126..."


Set `DISTRICT_NAME_COLUMN` below to whatever the actual district identifier column is called, based on the printed output above. This is left as an explicit variable rather than hardcoded further down so it only needs to be set once, in one place.

In [10]:
DISTRICT_NAME_COLUMN = "DistrictNa"  # placeholder, set from the printed columns above

Before joining, this is the point to check the known placeholder coordinate cluster directly. It will not show up as unmatched, since Hemet is a real place inside a real district polygon, it will show up as every one of those rows getting assigned the same single district regardless of the property's true location. Reporting this now means it is a documented, visible limitation of this feature rather than a silent one.

In [11]:
PLACEHOLDER_LAT = 33.694407
PLACEHOLDER_LON = -116.969959

placeholder_mask = (housing["Latitude"] == PLACEHOLDER_LAT) & (housing["Longitude"] == PLACEHOLDER_LON)
print(f"rows still on the known placeholder coordinate: {placeholder_mask.sum()}")
print("these rows will all be assigned whatever single district contains that point, "
      "regardless of each property's true location, until the confirmed geocoding "
      "pipeline output is merged in to replace this coordinate.")

rows still on the known placeholder coordinate: 107
these rows will all be assigned whatever single district contains that point, regardless of each property's true location, until the confirmed geocoding pipeline output is merged in to replace this coordinate.


In [12]:
points_gdf = gpd.GeoDataFrame(
    housing,
    geometry=gpd.points_from_xy(housing["Longitude"], housing["Latitude"]),
    crs="EPSG:4326",
)

if points_gdf.crs != districts_gdf.crs:
    points_gdf = points_gdf.to_crs(districts_gdf.crs)

joined = gpd.sjoin(
    points_gdf,
    districts_gdf[[DISTRICT_NAME_COLUMN, "geometry"]],
    how="left",
    predicate="within",
)
joined = joined.drop(columns=["geometry", "index_right"])

unmatched = joined[DISTRICT_NAME_COLUMN].isna().sum()
print(f"unmatched to any district: {unmatched} / {len(joined)} ({unmatched/len(joined):.1%})")
print("a high unmatched rate here is a signal to check, most likely a CRS mismatch "
      "or coordinates that fall outside every district polygon in the shapefile, "
      "not something to filter past without looking at it first.")

housing = pd.DataFrame(joined)

unmatched to any district: 199 / 401212 (0.0%)
a high unmatched rate here is a signal to check, most likely a CRS mismatch or coordinates that fall outside every district polygon in the shapefile, not something to filter past without looking at it first.


With a real district feature now joined in, the old raw `HighSchoolDistrict` text column is redundant and noisier than what replaces it, so it is dropped rather than kept alongside the new feature.

In [13]:
housing = housing.rename(columns={DISTRICT_NAME_COLUMN: "SchoolDistrictJoined"})
housing = housing.drop(columns=["HighSchoolDistrict"])
print("final columns:", list(housing.columns))

final columns: ['Flooring', 'ViewYN', 'PoolPrivateYN', 'ClosePrice', 'Latitude', 'Longitude', 'LivingArea', 'MLSAreaMajor', 'CountyOrParish', 'AttachedGarageYN', 'ParkingTotal', 'YearBuilt', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'FireplaceYN', 'Stories', 'Levels', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet', 'SaleYearMonth', 'PropertyAgeYears', 'BedBathRatio', 'SchoolDistrictJoined']


## 5. m2 Chronological Split and Outlier Filtering

Same helper function and same order of operations as m1: split first, then fit outlier thresholds on the m2 training set alone and apply the frozen thresholds to all three partitions. This is a fresh split on the enriched dataset, not a reuse of the m1 partitions, since row counts changed from the implausible value filtering above.

In [14]:
def chronological_train_val_test_split(df, period_col="SaleYearMonth", n_train_months=None):
    periods = sorted(df[period_col].dropna().unique())
    if len(periods) < 3:
        raise ValueError("Need at least three unique time periods.")
    test_period = periods[-1]
    val_period = periods[-2]
    train_periods = periods[:-2]
    if n_train_months is not None:
        train_periods = train_periods[-n_train_months:]
    train_df = df[df[period_col].isin(train_periods)].sort_values(period_col).reset_index(drop=True)
    val_df = df[df[period_col] == val_period].sort_values(period_col).reset_index(drop=True)
    test_df = df[df[period_col] == test_period].sort_values(period_col).reset_index(drop=True)
    return train_df, val_df, test_df

N_TRAIN_MONTHS = 12  # kept the same as m1 for a fair comparison

housingtrainm2, housingvalm2, housingtestm2 = chronological_train_val_test_split(
    housing, period_col="SaleYearMonth", n_train_months=N_TRAIN_MONTHS
)

In [15]:
LOWER_LIMIT_PERCENTILE = 0.5
UPPER_LIMIT_PERCENTILE = 99.5

lower_limit_m2 = housingtrainm2["ClosePrice"].quantile(LOWER_LIMIT_PERCENTILE / 100)
upper_limit_m2 = housingtrainm2["ClosePrice"].quantile(UPPER_LIMIT_PERCENTILE / 100)
print(f"m2 outlier thresholds (fit on m2 training data only): [{lower_limit_m2:,.0f}, {upper_limit_m2:,.0f}]")

def apply_outlier_thresholds(df, lower, upper, label):
    before = len(df)
    filtered = df[(df["ClosePrice"] > lower) & (df["ClosePrice"] < upper)]
    print(f"  {label}: {before} -> {len(filtered)} rows")
    return filtered

housingtrainm2 = apply_outlier_thresholds(housingtrainm2, lower_limit_m2, upper_limit_m2, "train")
housingvalm2 = apply_outlier_thresholds(housingvalm2, lower_limit_m2, upper_limit_m2, "val")
housingtestm2 = apply_outlier_thresholds(housingtestm2, lower_limit_m2, upper_limit_m2, "test")

m2 outlier thresholds (fit on m2 training data only): [185,000, 8,500,000]
  train: 161318 -> 159681 rows
  val: 15018 -> 14856 rows
  test: 15025 -> 14887 rows


In [16]:
os.makedirs("CRMLSCleaned", exist_ok=True)
housingtrainm2.to_csv("CRMLSCleaned/housingtrainm2.csv", index=False)
housingvalm2.to_csv("CRMLSCleaned/housingvalm2.csv", index=False)
housingtestm2.to_csv("CRMLSCleaned/housingtestm2.csv", index=False)
print("saved housingtrainm2 / housingvalm2 / housingtestm2")

saved housingtrainm2 / housingvalm2 / housingtestm2


## 6. Retrain on the m2 Feature Set

The old m1 models are also retrained here rather than pulled from notebook 4's saved numbers, so the comparison in the next section is computed under identical conditions in one run, not assembled by hand from two different notebooks.

In [20]:
def evaluate(pipeline, X, y, label):
    preds = pipeline.predict(X)
    r2 = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)
    mape = mean_absolute_percentage_error(y, preds)
    mdape = np.median(np.abs((y - preds) / y))
    print(f"{label:>5s}: R2={r2:.4f}  MAE=${mae:,.0f}  MAPE={mape:.2%}  MdAPE={mdape:.2%}")
    return {"r2": r2, "mae": mae, "mape": mape, "mdape": mdape}

def make_preprocessor(numeric_cols, categorical_cols):
    return ColumnTransformer(transformers=[
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
    ])

non_feature_columns_m2 = ["ClosePrice", "SaleYearMonth"]
numeric_feature_columns_m2 = [
    "Latitude", "Longitude",
    "ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "ParkingTotal", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "GarageSpaces",
    "LivingArea", "LotSizeSquareFeet", "AssociationFee", "YearBuilt", "Levels", "Stories",
    "PropertyAgeYears", "BedBathRatio",
]
categorical_feature_columns_m2 = ["City", "CountyOrParish", "MLSAreaMajor", "Flooring", "SchoolDistrictJoined"]
feature_columns_m2 = numeric_feature_columns_m2 + categorical_feature_columns_m2

assert set(feature_columns_m2) == set(housingtrainm2.columns) - set(non_feature_columns_m2), (
    "feature_columns_m2 doesn't match housingtrainm2's actual columns, check for a typo."
)

X_train_m2, y_train_m2 = housingtrainm2[feature_columns_m2], housingtrainm2["ClosePrice"]
X_val_m2, y_val_m2 = housingvalm2[feature_columns_m2], housingvalm2["ClosePrice"]
X_test_m2, y_test_m2 = housingtestm2[feature_columns_m2], housingtestm2["ClosePrice"]

results_m2 = {}

print("--- m2 Linear Regression ---")
lr_m2 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns_m2, categorical_feature_columns_m2)), ("model", LinearRegression())])
lr_m2.fit(X_train_m2, y_train_m2)
results_m2["LinearRegression"] = {
    "train": evaluate(lr_m2, X_train_m2, y_train_m2, "train"),
    "val": evaluate(lr_m2, X_val_m2, y_val_m2, "val"),
    "test": evaluate(lr_m2, X_test_m2, y_test_m2, "test"),
}

print("\n--- m2 Decision Tree ---")
dt_m2 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns_m2, categorical_feature_columns_m2)), ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))])
dt_m2.fit(X_train_m2, y_train_m2)
results_m2["DecisionTree"] = {
    "train": evaluate(dt_m2, X_train_m2, y_train_m2, "train"),
    "val": evaluate(dt_m2, X_val_m2, y_val_m2, "val"),
    "test": evaluate(dt_m2, X_test_m2, y_test_m2, "test"),
}

print("\n--- m2 Random Forest ---")
rf_m2 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns_m2, categorical_feature_columns_m2)), ("model", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))])
rf_m2.fit(X_train_m2, y_train_m2)
results_m2["RandomForest"] = {
    "train": evaluate(rf_m2, X_train_m2, y_train_m2, "train"),
    "val": evaluate(rf_m2, X_val_m2, y_val_m2, "val"),
    "test": evaluate(rf_m2, X_test_m2, y_test_m2, "test"),
}

--- m2 Linear Regression ---
train: R2=0.8387  MAE=$218,989  MAPE=20.75%  MdAPE=14.86%
  val: R2=0.8337  MAE=$235,141  MAPE=21.46%  MdAPE=15.02%
 test: R2=0.8326  MAE=$235,442  MAPE=21.18%  MdAPE=15.23%

--- m2 Decision Tree ---
train: R2=1.0000  MAE=$137  MAPE=0.01%  MdAPE=0.00%
  val: R2=0.7677  MAE=$229,533  MAPE=16.83%  MdAPE=10.57%
 test: R2=0.7712  MAE=$230,386  MAPE=16.39%  MdAPE=10.71%

--- m2 Random Forest ---
train: R2=0.9887  MAE=$44,735  MAPE=3.44%  MdAPE=1.87%
  val: R2=0.8866  MAE=$163,043  MAPE=12.04%  MdAPE=7.67%
 test: R2=0.8804  MAE=$165,006  MAPE=11.90%  MdAPE=7.69%


## 7. Retrain the m1 Feature Set for Comparison

Loaded from the saved m1 CSVs and run through the exact same three models, so the table in the next section reflects the feature set difference and nothing else. Sample size will differ slightly between m1 and m2 given the implausible value filtering above, that difference is itself part of what changed, not something to normalize away.

In [18]:
housingtrainm1 = pd.read_csv("CRMLSCleaned/housingtrainm1.csv")
housingvalm1 = pd.read_csv("CRMLSCleaned/housingvalm1.csv")
housingtestm1 = pd.read_csv("CRMLSCleaned/housingtestm1.csv")

non_feature_columns_m1 = ["ClosePrice", "SaleYearMonth"]
numeric_feature_columns_m1 = [
    "Latitude", "Longitude",
    "ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "ParkingTotal", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "GarageSpaces",
    "LivingArea", "LotSizeSquareFeet", "AssociationFee", "YearBuilt", "Levels", "Stories",
]
categorical_feature_columns_m1 = ["City", "CountyOrParish", "MLSAreaMajor", "HighSchoolDistrict", "Flooring"]
feature_columns_m1 = numeric_feature_columns_m1 + categorical_feature_columns_m1

X_train_m1, y_train_m1 = housingtrainm1[feature_columns_m1], housingtrainm1["ClosePrice"]
X_val_m1, y_val_m1 = housingvalm1[feature_columns_m1], housingvalm1["ClosePrice"]
X_test_m1, y_test_m1 = housingtestm1[feature_columns_m1], housingtestm1["ClosePrice"]

results_m1 = {}

print("--- m1 Linear Regression ---")
lr_m1 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns_m1, categorical_feature_columns_m1)), ("model", LinearRegression())])
lr_m1.fit(X_train_m1, y_train_m1)
results_m1["LinearRegression"] = {"test": evaluate(lr_m1, X_test_m1, y_test_m1, "test")}

print("\n--- m1 Decision Tree ---")
dt_m1 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns_m1, categorical_feature_columns_m1)), ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))])
dt_m1.fit(X_train_m1, y_train_m1)
results_m1["DecisionTree"] = {"test": evaluate(dt_m1, X_test_m1, y_test_m1, "test")}

print("\n--- m1 Random Forest ---")
rf_m1 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns_m1, categorical_feature_columns_m1)), ("model", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))])
rf_m1.fit(X_train_m1, y_train_m1)
results_m1["RandomForest"] = {"test": evaluate(rf_m1, X_test_m1, y_test_m1, "test")}

--- m1 Linear Regression ---
 test: R2=0.8217  MAE=$248,884  MAPE=22.68%  MdAPE=16.10%

--- m1 Decision Tree ---
 test: R2=0.7824  MAE=$232,524  MAPE=16.91%  MdAPE=11.19%

--- m1 Random Forest ---
 test: R2=0.8768  MAE=$169,785  MAPE=12.24%  MdAPE=7.89%


## 8. Old vs New Feature Set Comparison

In [19]:
comparison_rows = []
for model_name in ["LinearRegression", "DecisionTree", "RandomForest"]:
    comparison_rows.append({
        "model": model_name,
        "feature_set": "m1 (original)",
        "test_r2": results_m1[model_name]["test"]["r2"],
        "test_mae": results_m1[model_name]["test"]["mae"],
        "test_mape": results_m1[model_name]["test"]["mape"],
        "test_mdape": results_m1[model_name]["test"]["mdape"],
    })
    comparison_rows.append({
        "model": model_name,
        "feature_set": "m2 (engineered)",
        "test_r2": results_m2[model_name]["test"]["r2"],
        "test_mae": results_m2[model_name]["test"]["mae"],
        "test_mape": results_m2[model_name]["test"]["mape"],
        "test_mdape": results_m2[model_name]["test"]["mdape"],
    })

comparison_df = pd.DataFrame(comparison_rows).set_index(["model", "feature_set"])
comparison_df["test_r2_change"] = comparison_df.groupby("model")["test_r2"].diff()
comparison_df

test_r2       test_mae  test_mape  \
model            feature_set                                           
LinearRegression m1 (original)    0.821687  248883.899745   0.226760   
                 m2 (engineered)  0.832561  235442.424872   0.211786   
DecisionTree     m1 (original)    0.782358  232524.240549   0.169051   
                 m2 (engineered)  0.771247  230386.387291   0.163931   
RandomForest     m1 (original)    0.876843  169784.611675   0.122388   
                 m2 (engineered)  0.880421  165006.114771   0.119016   

                                  test_mdape  test_r2_change  
model            feature_set                                  
LinearRegression m1 (original)      0.161032             NaN  
                 m2 (engineered)    0.152311        0.010874  
DecisionTree     m1 (original)      0.111896             NaN  
                 m2 (engineered)    0.107059       -0.011111  
RandomForest     m1 (original)      0.078883             NaN  
                 m2 (engineered)    0.076890        0.003579

### Feature Engineering & Model Evaluation

1. **Added new engineered features.** I expanded the baseline dataset by creating **PropertyAgeYears** (sale year − year built), **BedBathRatio** (bedrooms divided by bathrooms), and **SchoolDistrictJoined** through a spatial join with California school district boundaries. These features were intended to capture property age, home layout, and geographic influences on housing prices.

2. **Evaluated automatic bedroom and bathroom cutoffs.** I tested an automated frequency-based approach for identifying outliers in bedroom and bathroom counts. The suggested cutoffs (3 bathrooms and 5 bedrooms) would have removed a substantial number of legitimate homes, so I decided not to apply these filters and instead preserved the original distributions.

3. **Improved overall model performance.** After retraining the models with the engineered features, both **Linear Regression** and **Random Forest** achieved higher test performance, with Random Forest remaining the strongest model (**R² = 0.8804**). The Decision Tree continued to overfit the training data and experienced a slight decrease in test performance, indicating that the engineered features benefited more robust models than a single unpruned tree.